# LAB 2 — Preprocessing and Traditional Language Representation using scikit-learn

**Natural Language and Information Retrieval**  
**Degree in Data Science**  
**ETSINF — Universitat Politècnica de València**


## Authors

- Sergio Ortiz
- Fernando Martínez


In [1]:
# IF YOU USE GOOGLE COLAB AND WANT READ A FILE
#from google.colab import drive
#drive.mount('/content/drive')

## Goal of the notebook

This notebook solves **Lab Session 2** on preprocessing and traditional text representation.

We follow the required pipeline:

1. Load the dataset.
2. Assign one final label to each meme using **majority voting**.
3. Apply the required preprocessing.
4. Build the seven requested text representations.
5. Compute cosine similarities.
6. Retrieve the most similar pair **inside each class** (`SEXIST` and `NON-SEXIST`) for every representation.

## Notation

Let

$$
\mathcal{D}=\{d_1,d_2,\dots,d_N\}
$$

be the corpus of memes.

For each meme $d_i$, let

$$
A_i=(a_{i1},a_{i2},\dots,a_{i6}), \qquad a_{ij}\in\{\mathrm{YES},\mathrm{NO}\}
$$

be the six annotations stored in `labels_task2_1`.

We define the number of positive and negative votes as

$$
Y_i=\sum_{j=1}^{6}\mathbf{1}(a_{ij}=\mathrm{YES}),
\qquad
N_i=\sum_{j=1}^{6}\mathbf{1}(a_{ij}=\mathrm{NO}).
$$

Then the final class $c_i$ is assigned by majority voting:

$$
c_i=
\begin{cases}
\mathrm{SEXIST}, & \text{if } Y_i\ge 4,\\[4pt]
\mathrm{NON\text{-}SEXIST}, & \text{if } N_i\ge 4.
\end{cases}
$$

This is exactly the rule requested in the lab statement.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import json
import pandas as pd
import re

definition = {
    "a": "Bag of Words (binary)",
    "b": "Bag of Words (Term-Frequency) without normalization",
    "c": "Bigrams of Words (Term-Frequency) without normalization",
    "d": "Trigram of Characters (Term-Frequency) without normalization",
    "e": "TF-IDF based on words with 'l2' normalization",
    "f": "LSA based on TF-IDF of words (50 singular values)",
    "g": "LSA based on TF-IDF of words (100 singular values)"
}
# !pip install pandas

## Load the dataset

In [4]:
from pathlib import Path

data_path = Path("EXIST2026_training_subset.json") / "EXIST2026_training_subset.json"

with data_path.open("r", encoding="utf-8") as f:
    raw_data = json.load(f)

rows = []
for _, item in raw_data.items():
    labels = item["labels_task2_1"]
    yes_votes = sum(label == "YES" for label in labels)
    no_votes = sum(label == "NO" for label in labels)

    if yes_votes >= 4:
        final_label = "SEXIST"
    elif no_votes >= 4:
        final_label = "NON-SEXIST"
    else:
        final_label = "UNDECIDED"

    rows.append({
        "id_EXIST": item["id_EXIST"],
        "text": item["text"],
        "labels_task2_1": labels,
        "yes_votes": yes_votes,
        "no_votes": no_votes,
        "label": final_label
    })

df = pd.DataFrame(rows)

df.head()

,id_EXIST,text,labels_task2_1,yes_votes,no_votes,label
0,210087,LADIES PLEASE CONTAIN YOUR ORGASM,"[NO, NO, NO, NO, NO, YES]",1,5,NON-SEXIST
1,210243,Me seeing boobs (age 15) Me seeing boobs (age ...,"[YES, NO, YES, NO, YES, YES]",4,2,SEXIST
2,210530,WWIII *begins* Feminists:,"[YES, NO, YES, YES, YES, YES]",5,1,SEXIST
3,211995,men should get vasectomies as a form of birth ...,"[NO, NO, NO, NO, NO, YES]",1,5,NON-SEXIST
4,211239,"WOMEN AREN'T OBJECTS? FALSE, AN OBJECT IS ANYT...","[YES, YES, YES, NO, YES, YES]",5,1,SEXIST


### Labeling analysis

The dataset contains one row per meme. Each row includes six human annotations in `labels_task2_1`.

After applying majority voting, we convert the original list of six decisions into one final binary label:

- `SEXIST`
- `NON-SEXIST`

This final label is the target variable used in the rest of the notebook.


## Preprocess dataset

In [5]:

url_pattern = re.compile(r"https?://\S+|www\.\S+")

def preprocess(text):
    """
    Required preprocessing for the lab:
    1. remove URLs
    2. lowercase the text
    """
    if pd.isna(text):
        return ""
    text = str(text)
    text = url_pattern.sub("", text)
    text = text.lower().strip()
    return text


df["text_preprocessed"] = df["text"].apply(preprocess)

df[["id_EXIST", "text", "text_preprocessed", "label"]].head()


,id_EXIST,text,text_preprocessed,label
0,210087,LADIES PLEASE CONTAIN YOUR ORGASM,ladies please contain your orgasm,NON-SEXIST
1,210243,Me seeing boobs (age 15) Me seeing boobs (age ...,me seeing boobs (age 15) me seeing boobs (age ...,SEXIST
2,210530,WWIII *begins* Feminists:,wwiii *begins* feminists:,SEXIST
3,211995,men should get vasectomies as a form of birth ...,men should get vasectomies as a form of birth ...,NON-SEXIST
4,211239,"WOMEN AREN'T OBJECTS? FALSE, AN OBJECT IS ANYT...","women aren't objects? false, an object is anyt...",SEXIST


### Preprocessing analysis

Let $T_i$ be the original text of meme $d_i$.

We define its preprocessed version as

$$
T_i'=\operatorname{lowercase}\!\big(\operatorname{removeURL}(T_i)\big).
$$

This is the **minimum preprocessing** explicitly required by the lab statement.

In other words:

1. URLs are removed.
2. The remaining text is converted to lowercase.

No stemming, lemmatization, or stopword removal is applied here, because those operations are **not mandatory** in the exercise.


## Create the vectorizers and compute the text representations

In [6]:

texts = df["text_preprocessed"].tolist()

# a) Bag of Words (binary)
vectorizer_a = CountVectorizer(binary=True, preprocessor=None)
X_a = vectorizer_a.fit_transform(texts)

# b) Bag of Words (term frequency)
vectorizer_b = CountVectorizer(binary=False, preprocessor=None)
X_b = vectorizer_b.fit_transform(texts)

# c) Bigrams of words (term frequency)
vectorizer_c = CountVectorizer(ngram_range=(2, 2), analyzer="word", binary=False, preprocessor=None)
X_c = vectorizer_c.fit_transform(texts)

# d) Trigrams of characters (term frequency)
vectorizer_d = CountVectorizer(ngram_range=(3, 3), analyzer="char", binary=False, preprocessor=None)
X_d = vectorizer_d.fit_transform(texts)

# e) TF-IDF based on words with l2 normalization
vectorizer_e = TfidfVectorizer(norm="l2", use_idf=True, preprocessor=None)
X_e = vectorizer_e.fit_transform(texts)

# f) LSA with 50 singular values over TF-IDF
svd_50 = TruncatedSVD(n_components=50, random_state=42)
X_f = svd_50.fit_transform(X_e)

# g) LSA with 100 singular values over TF-IDF
svd_100 = TruncatedSVD(n_components=100, random_state=42)
X_g = svd_100.fit_transform(X_e)

representations = {
    "a": X_a,
    "b": X_b,
    "c": X_c,
    "d": X_d,
    "e": X_e,
    "f": X_f,
    "g": X_g,
}

for key, matrix in representations.items():
    print(f"{key}) {definition[key]} -> shape = {matrix.shape}")


a) Bag of Words (binary) -> shape = (1000, 4429)
b) Bag of Words (Term-Frequency) without normalization -> shape = (1000, 4429)
c) Bigrams of Words (Term-Frequency) without normalization -> shape = (1000, 13700)
d) Trigram of Characters (Term-Frequency) without normalization -> shape = (1000, 8018)
e) TF-IDF based on words with 'l2' normalization -> shape = (1000, 4429)
f) LSA based on TF-IDF of words (50 singular values) -> shape = (1000, 50)
g) LSA based on TF-IDF of words (100 singular values) -> shape = (1000, 100)


### Representation analysis

For each meme $d_i$, we build a vector representation

$$
x_i^{(r)} \in \mathbb{R}^{m_r},
$$

where $r$ denotes the selected representation and $m_r$ is its dimensionality.

#### a) Binary Bag of Words

Let $t_j$ be the $j$-th term in the vocabulary. Then

$$
x_{ij}^{(a)}=
\begin{cases}
1, & \text{if } t_j \text{ appears in } d_i,\\[4pt]
0, & \text{otherwise}.
\end{cases}
$$

This representation captures only **presence or absence** of words.

#### b) Term-Frequency Bag of Words

$$
x_{ij}^{(b)} = f(t_j,d_i),
$$

where $f(t_j,d_i)$ is the number of occurrences of term $t_j$ in document $d_i$.

#### c) Word bigrams

The features are adjacent pairs of words:

$$
(w_1,w_2),\ (w_2,w_3),\ \dots,\ (w_{n-1},w_n).
$$

This introduces **local word order** into the representation.

#### d) Character trigrams

The features are sequences of three consecutive characters:

$$
(c_1c_2c_3),\ (c_2c_3c_4),\ \dots,\ (c_{n-2}c_{n-1}c_n).
$$

This is useful for noisy text, spelling variation, and short informal messages.

#### e) TF-IDF

The TF-IDF weight of term $t$ in document $d$ is conceptually defined as

$$
\operatorname{tfidf}(t,d)=\operatorname{tf}(t,d)\cdot \operatorname{idf}(t).
$$

It increases with the importance of the term inside the document and decreases when the term appears in many documents of the corpus.

#### f) and g) LSA

LSA applies truncated SVD to the TF-IDF matrix and produces lower-dimensional latent representations:

$$
X_{\mathrm{LSA50}} \in \mathbb{R}^{N\times 50},
\qquad
X_{\mathrm{LSA100}} \in \mathbb{R}^{N\times 100}.
$$

These spaces preserve the main latent semantic directions while reducing dimensionality.


## Compute cosine similarities

In [7]:

def most_similar_pair(X, subset_df):
    """
    Return the most similar pair of different samples inside subset_df.
    The diagonal is ignored because each document is trivially identical to itself.
    """
    idx = subset_df.index.to_numpy()
    X_subset = X[idx]
    sim_matrix = cosine_similarity(X_subset)

    # Ignore self-similarity
    sim_matrix[range(len(idx)), range(len(idx))] = -1

    i, j = divmod(sim_matrix.argmax(), sim_matrix.shape[1])

    row_i = subset_df.iloc[i]
    row_j = subset_df.iloc[j]

    return {
        "id1": row_i["id_EXIST"],
        "text1": row_i["text"],
        "id2": row_j["id_EXIST"],
        "text2": row_j["text"],
        "cosine_similarity": float(sim_matrix[i, j])
    }

results = []
for rep_name, X in representations.items():
    for label in ["SEXIST", "NON-SEXIST"]:
        subset_df = df[df["label"] == label]
        best = most_similar_pair(X, subset_df)
        results.append({
            "representation_id": rep_name,
            "representation": definition[rep_name],
            "class": label,
            **best
        })

results_df = pd.DataFrame(results)
results_df


,representation_id,representation,class,id1,text1,id2,text2,cosine_similarity
0,a,Bag of Words (binary),SEXIST,211469,Netflix: are you still watching? Somebody's da...,211199,Netflix: Are you still watching? Someone's Dau...,0.755929
1,a,Bag of Words (binary),NON-SEXIST,210206,I DON'T BLAME DISNEY FOR THE MOST handsome ACT...,210203,I DON'T BLAME DISNEY FOR MY HIGH EXPECTATIONS ...,0.648204
2,b,Bag of Words (Term-Frequency) without normaliz...,SEXIST,211021,mgflip.com psycosis autism Corporate needs you...,211897,Third Wave Feminism Corporate needs you to fin...,0.831655
3,b,Bag of Words (Term-Frequency) without normaliz...,NON-SEXIST,211110,When you see your girl naked for the 1st. time...,211091,when you see your girl naked MEMES,0.717137
4,c,Bigrams of Words (Term-Frequency) without norm...,SEXIST,211021,mgflip.com psycosis autism Corporate needs you...,211897,Third Wave Feminism Corporate needs you to fin...,0.682242
5,c,Bigrams of Words (Term-Frequency) without norm...,NON-SEXIST,211110,When you see your girl naked for the 1st. time...,211091,when you see your girl naked MEMES,0.690066
6,d,Trigram of Characters (Term-Frequency) without...,SEXIST,211021,mgflip.com psycosis autism Corporate needs you...,211897,Third Wave Feminism Corporate needs you to fin...,0.864754
7,d,Trigram of Characters (Term-Frequency) without...,NON-SEXIST,211110,When you see your girl naked for the 1st. time...,211091,when you see your girl naked MEMES,0.738095
8,e,TF-IDF based on words with 'l2' normalization,SEXIST,211469,Netflix: are you still watching? Somebody's da...,211199,Netflix: Are you still watching? Someone's Dau...,0.732429
9,e,TF-IDF based on words with 'l2' normalization,NON-SEXIST,211110,When you see your girl naked for the 1st. time...,211091,when you see your girl naked MEMES,0.686520


### Similarity analysis

Given two document vectors $x$ and $y$, cosine similarity is

$$
\cos(x,y)=\frac{x\cdot y}{\lVert x\rVert\,\lVert y\rVert}.
$$

This metric measures how aligned two document vectors are.

- A value close to $1$ means the vectors are very similar.
- A value close to $0$ means they are weakly related in that representation space.

For each representation, we compute the pairwise cosine similarity matrix **inside each class separately**:

- once over the set of `SEXIST` memes,
- once over the set of `NON-SEXIST` memes.

Then we select the pair with the **largest non-diagonal value**.


## Show results

In [8]:

summary_df = results_df[["representation_id", "representation", "class", "id1", "id2", "cosine_similarity"]].copy()
summary_df = summary_df.sort_values(["representation_id", "class"]).reset_index(drop=True)
summary_df


,representation_id,representation,class,id1,id2,cosine_similarity
0,a,Bag of Words (binary),NON-SEXIST,210206,210203,0.648204
1,a,Bag of Words (binary),SEXIST,211469,211199,0.755929
2,b,Bag of Words (Term-Frequency) without normaliz...,NON-SEXIST,211110,211091,0.717137
3,b,Bag of Words (Term-Frequency) without normaliz...,SEXIST,211021,211897,0.831655
4,c,Bigrams of Words (Term-Frequency) without norm...,NON-SEXIST,211110,211091,0.690066
5,c,Bigrams of Words (Term-Frequency) without norm...,SEXIST,211021,211897,0.682242
6,d,Trigram of Characters (Term-Frequency) without...,NON-SEXIST,211110,211091,0.738095
7,d,Trigram of Characters (Term-Frequency) without...,SEXIST,211021,211897,0.864754
8,e,TF-IDF based on words with 'l2' normalization,NON-SEXIST,211110,211091,0.686520
9,e,TF-IDF based on words with 'l2' normalization,SEXIST,211469,211199,0.732429


## Final discussion

The most similar pair depends on the selected representation because each representation preserves different information:

- **Binary BoW** emphasizes lexical overlap in terms of presence or absence.
- **Term-frequency BoW** also captures repeated words.
- **Word bigrams** capture short local word-order patterns.
- **Character trigrams** are robust to spelling variation and short noisy expressions.
- **TF-IDF** downweights very common terms in the corpus and emphasizes more informative ones.
- **LSA** projects the texts into a latent semantic space, which often produces very high similarity values for near-duplicate patterns.

Therefore, the representation is not just a technical detail: it defines what we consider **similar** in the corpus.
